# 00 — Data Preparation

**One-time notebook. Run once, outputs are cached — re-running skips completed steps.**

Outputs written to `data/`:
- `signals.parquet` — cleaned signal rows (non-delete, Fluff/Filler dropped, float32)
- `universes.json` — SP500 / SP1500 / RU3K ticker lists
- `prices.parquet` — daily adj-close for all universe tickers (yfinance, auto_adjust=True)
- `events_with_returns.parquet` — 376,790 Total-slice events with entry dates, 5 forward returns, and 772 engineered features (193 base Aspect×Theme cross-product + QoQ/2Q/YoY trends)
- `sparse_features.parquet` — raw 405 AspectTheme columns for the Stretch model

**Look-ahead audit checklist:**
- Entry date: hour ≥ 16 UTC → next NYSE trading day; hour < 16 UTC → same day
- Forward returns are targets only — asserted absent from feature column set (cell 25)
- QoQ / 2Q / YoY deltas use `shift(1/2/4)` within ticker sorted by entry_date (past data only)
- Cross-sectional ranks deferred to analysis notebook (computed per walk-forward fold)
- Fluff/Filler columns dropped at CSV-load time — never written to Parquet
- Returns winsorized at 0.1%/99.9% using **quarterly expanding-window bounds** (prior quarters only, look-ahead-free)
- NaN-return events excluded — no roll-forward applied
- Date-only rows (hour = 0): treated as BMO → same-day entry (conservative, documented)

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import yfinance as yf
from pathlib import Path
from tqdm.notebook import tqdm
import json, warnings, time

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

BASE_DIR  = Path('.')
DATA_DIR  = BASE_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)

RAW_CSV = BASE_DIR / 'Earnings_ATC_until_2026-04-21.csv'
print(f'Raw CSV: {RAW_CSV}  ({RAW_CSV.stat().st_size / 1e9:.2f} GB)')

/Users/yueqilin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Raw CSV: Earnings_ATC_until_2026-04-21.csv  (4.47 GB)


## 2. Column Inspection

In [2]:
# Read header only — zero data loaded
all_cols = list(pd.read_csv(RAW_CSV, nrows=0).columns)

# Cols 0-41: identifiers + 12 EventScores + ATCClassifierScore
id_signal_cols = all_cols[:42]

# AspectTheme columns: drop Fluff and Filler (noise classes by design)
at_cols_all   = [c for c in all_cols if c.startswith('AspectTheme_')]
fluff_filler  = [c for c in at_cols_all if 'Fluff' in c or 'Filler' in c]
at_cols_keep  = [c for c in at_cols_all if c not in set(fluff_filler)]

keep_cols = id_signal_cols + at_cols_keep

print(f'Total CSV columns       : {len(all_cols)}')
print(f'  id/signal cols (0-41) : {len(id_signal_cols)}')
print(f'  AspectTheme all       : {len(at_cols_all)}')
print(f'  Fluff/Filler dropped  : {len(fluff_filler)}')
print(f'  AspectTheme kept      : {len(at_cols_keep)}')
print(f'Columns to load         : {len(keep_cols)}')
print()
print('Key signal columns:', id_signal_cols[28:42])

Total CSV columns       : 609
  id/signal cols (0-41) : 42
  AspectTheme all       : 567
  Fluff/Filler dropped  : 162
  AspectTheme kept      : 405
Columns to load         : 447

Key signal columns: ['Sentences', 'EventPos_1_1_1', 'EventNeg_1_1_1', 'EventsScore_1_1_1', 'EventPos_4_2_1', 'EventNeg_4_2_1', 'EventsScore_4_2_1', 'EventPos_3_1_0', 'EventNeg_3_1_0', 'EventsScore_3_1_0', 'EventPos_1_1_0', 'EventNeg_1_1_0', 'EventsScore_1_1_0', 'ATCClassifierScore']


In [3]:
# Parse AspectTheme column metadata
# Format: AspectTheme_{Aspect}_{Theme} - {Magnitude} - {Sentiment}
def parse_at_col(col):
    rest  = col.replace('AspectTheme_', '')
    parts = rest.split(' - ')
    aspect_theme = parts[0]
    magnitude    = parts[1]
    sentiment    = parts[2]
    aspect, theme = aspect_theme.split('_', 1)
    return aspect, theme, magnitude, sentiment

at_meta = pd.DataFrame(
    [parse_at_col(c) + (c,) for c in at_cols_keep],
    columns=['aspect', 'theme', 'magnitude', 'sentiment', 'col']
)

print('Aspects :', sorted(at_meta['aspect'].unique()))
print('Themes  :', sorted(at_meta['theme'].unique()))
print('Magnitudes:', sorted(at_meta['magnitude'].unique()))
print('Sentiments:', sorted(at_meta['sentiment'].unique()))
print(f'\n{len(at_meta)} valid AspectTheme cells')

Aspects : ['CurrentState', 'Forecast', 'Other', 'StrategicPosition', 'Surprise']
Themes  : ['CapitalAllocation', 'ESG', 'FinancialPerformance', 'MacroeconomicFactors', 'MarketAndCompetitivePosition', 'OperationalPerformance', 'Other', 'RegulatoryAndLegalIssues', 'StrategicInitiatives']
Magnitudes: ['High', 'Low', 'Medium']
Sentiments: ['Negative', 'Neutral', 'Positive']

405 valid AspectTheme cells


## 3. Load CSV → Parquet  
Streams in 100k-row chunks; never holds the full CSV in RAM.

In [4]:
SIGNALS_PARQUET = DATA_DIR / 'signals.parquet'
CHUNK_SIZE = 100_000

if SIGNALS_PARQUET.exists():
    print(f'signals.parquet already exists ({SIGNALS_PARQUET.stat().st_size / 1e9:.2f} GB) — skipping.')
else:
    print('Loading CSV in chunks → writing Parquet...')
    writer        = None
    total_written = 0

    for i, chunk in enumerate(
        pd.read_csv(RAW_CSV, chunksize=CHUNK_SIZE, usecols=keep_cols, low_memory=False)
    ):
        chunk = chunk[chunk['SignalType'] != 'delete'].copy()

        # Cast ALL numeric columns to float32 up-front.
        # Using only float64 would miss columns that pandas reads as int64 when a chunk
        # has no NaN but float64 (→ float32) when a later chunk does — causing a schema
        # mismatch error in the ParquetWriter.
        for col in chunk.select_dtypes(['int64', 'float64']).columns:
            chunk[col] = chunk[col].astype('float32')

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(SIGNALS_PARQUET, table.schema, compression='snappy')
        writer.write_table(table)

        total_written += len(chunk)
        if (i + 1) % 5 == 0:
            print(f'  {total_written:>10,} rows written...')

    if writer:
        writer.close()
    print(f'\nDone. {total_written:,} rows → {SIGNALS_PARQUET.stat().st_size / 1e9:.2f} GB Parquet')

signals.parquet already exists (0.32 GB) — skipping.


## 4. EDA on Parquet

In [5]:
# Load identifier columns only — fast, small
eda_cols = [
    'SignalType', 'DocDate', 'BESTTICKER', 'SECTOR', 'COUNTRY',
    'EXCHANGE', 'QTR_YEAR', 'MOSTIMPORTANTDATEUTC', 'ATCClassifierScore'
]
df_meta = pd.read_parquet(SIGNALS_PARQUET, columns=eda_cols)

print(f'Shape: {df_meta.shape}')
print(f'\nSignalType distribution:')
print(df_meta['SignalType'].value_counts())
print(f'\nDate range: {df_meta["DocDate"].min()} → {df_meta["DocDate"].max()}')
print(f'Unique tickers: {df_meta["BESTTICKER"].nunique():,}')
print(f'\nTop 10 countries:')
print(df_meta['COUNTRY'].value_counts().head(10))
print(f'\nExchanges (to identify US stocks):')
print(df_meta['EXCHANGE'].value_counts().head(15))
print(f'\nATCClassifierScore stats (Total slice):')
print(df_meta[df_meta['SignalType']=='Total']['ATCClassifierScore'].describe())

Shape: (2738206, 9)

SignalType distribution:
SignalType
Total           376790
Executives      376036
Presentation    373808
Answer          359535
Question        351494
Analysts        343534
CEO             303854
CFO             253155
Name: count, dtype: int64



Date range: 2010-01-04 → 2026-04-21
Unique tickers: 17,636

Top 10 countries:
COUNTRY
United States     1496765
Canada             198706
India              147258
United Kingdom      90656
Australia           68017
Sweden              62606
China               58048
Japan               54577
Germany             53886
Brazil              48315
Name: count, dtype: int64

Exchanges (to identify US stocks):
EXCHANGE
NYSE        820265
NasdaqGS    440293
NasdaqGM    229600
NSEI        140988
TSX         137376
NasdaqCM    125629
OTCPK        91880
ASX          68575
LSE          67358
OM           63751
TSE          52015
XTRA         50821
BOVESPA      36504
NYSEAM       32191
OB           31669
Name: count, dtype: int64

ATCClassifierScore stats (Total slice):


count   376790.0000
mean         0.0153
std          0.0326
min         -0.2325
25%         -0.0022
50%          0.0152
75%          0.0323
max          0.2865
Name: ATCClassifierScore, dtype: float64


## 5. Universe Definitions

**Survivorship bias caveat**: All three universe lists reflect current composition (as of 2026).  
Historical constituents are not available (no CRSP/Compustat access).  
Reported alpha should be treated as an **upper bound** for each universe.  
This will be explicitly noted in the research PDF.

In [6]:
import requests

UNIVERSES_JSON = DATA_DIR / 'universes.json'

if UNIVERSES_JSON.exists():
    with open(UNIVERSES_JSON) as f:
        universes = json.load(f)
    sp500_tickers  = set(universes['SP500'])
    sp1500_tickers = set(universes['SP1500'])
    ru3k_tickers   = set(universes['RU3K'])
    print('Loaded universes.json')
else:
    # Wikipedia blocks Python's default urllib user-agent with 403 — use requests instead
    _hdrs = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120 Safari/537.36'}

    def _wiki_tickers(url):
        html    = requests.get(url, headers=_hdrs, timeout=30).text
        tbl     = pd.read_html(html)[0]
        sym_col = next(c for c in tbl.columns if 'ymbol' in str(c) or 'icker' in str(c))
        return set(tbl[sym_col].astype(str).str.replace('.', '-', regex=False))

    sp500_tickers  = _wiki_tickers('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
    sp400_tickers  = _wiki_tickers('https://en.wikipedia.org/wiki/List_of_S%26P_400_companies')
    sp600_tickers  = _wiki_tickers('https://en.wikipedia.org/wiki/List_of_S%26P_600_companies')
    sp1500_tickers = sp500_tickers | sp400_tickers | sp600_tickers

    # Exchange names as they actually appear in this dataset (see EDA output)
    US_EXCHANGES = {
        'NYSE', 'NasdaqGS', 'NasdaqGM', 'NasdaqCM',
        'NYSEAM', 'BATS', 'NYSE ARCA', 'NYSE MKT',
    }
    # dropna() removes rows where BESTTICKER is NaN — sorted() can't handle None vs str
    ru3k_tickers = set(
        df_meta[df_meta['EXCHANGE'].isin(US_EXCHANGES)]['BESTTICKER'].dropna().unique()
    )

    universes = {
        'SP500' : sorted(sp500_tickers),
        'SP1500': sorted(sp1500_tickers),
        'RU3K'  : sorted(ru3k_tickers),
    }
    with open(UNIVERSES_JSON, 'w') as f:
        json.dump(universes, f)
    print('Saved universes.json')

print(f'SP500  : {len(sp500_tickers):>5} tickers')
print(f'SP1500 : {len(sp1500_tickers):>5} tickers')
print(f'RU3K   : {len(ru3k_tickers):>5} tickers')

sig_tickers = set(df_meta['BESTTICKER'].dropna().unique())
for name, uni in [('SP500', sp500_tickers), ('SP1500', sp1500_tickers), ('RU3K', ru3k_tickers)]:
    print(f'  {name} tickers in signals: {len(uni & sig_tickers)} / {len(uni)}')

Loaded universes.json
SP500  :   503 tickers
SP1500 :  1506 tickers
RU3K   :  7877 tickers


  SP500 tickers in signals: 496 / 503
  SP1500 tickers in signals: 1463 / 1506
  RU3K tickers in signals: 7877 / 7877


## 6. Trading Calendar via SPY

In [7]:
# Download SPY to get the exact NYSE trading calendar (retry on rate-limit)
PRICE_START = '2009-12-01'
PRICE_END   = '2026-05-01'

spy_raw = None
for attempt in range(1, 6):
    spy_raw = yf.download('SPY', start=PRICE_START, end=PRICE_END,
                          auto_adjust=True, progress=False)
    if not spy_raw.empty:
        break
    print(f'SPY download attempt {attempt} failed (rate limit?) — waiting 15 s...')
    time.sleep(15)

if spy_raw is None or spy_raw.empty:
    raise RuntimeError('SPY download failed after 5 attempts. Check your internet connection.')

trading_days     = spy_raw.index.normalize()
trading_days_arr = np.array(trading_days, dtype='datetime64[D]')

print(f'NYSE trading days: {len(trading_days_arr):,}')
print(f'  From : {trading_days_arr[0]}')
print(f'  To   : {trading_days_arr[-1]}')

NYSE trading days: 4,128
  From : 2009-12-01
  To   : 2026-04-30


## 7. Fetch Prices via yfinance

Downloads adjusted-close prices for all tickers in any universe that also appear in the signals dataset.  
SP500/SP1500 tickers fetched in batches of 20 with individual retry for any misses; RU3K-only tickers fetched in batches of 50 (no retry — some micro-caps are expected to miss). Rate-limited at 0.5 s/batch. First run: ~45–60 min. Subsequent runs are incremental (skips already-fetched tickers).

In [8]:
PRICES_PARQUET = DATA_DIR / 'prices.parquet'

# ── Load already-fetched tickers (incremental: skip tickers we already have) ──
if PRICES_PARQUET.exists():
    _chk = pd.read_parquet(PRICES_PARQUET, columns=['ticker'])
    fetched_tickers = set(_chk['ticker'].unique())
    del _chk
    print(f'Existing prices.parquet: {len(fetched_tickers):,} tickers already fetched')
else:
    fetched_tickers = set()

# ── Fuzzy universe → BESTTICKER matching ─────────────────────────────────────
# Wikipedia normalizes "." → "-" (e.g. BRK.B → BRK-B) but BESTTICKER may use "."
# Strip both separators for matching, then use the BESTTICKER form for storage.
_nrm   = lambda t: t.replace('.','').replace('-','').upper()
_n2sig = {_nrm(t): t for t in sig_tickers}

def _match_uni(uni):
    out = set()
    for t in uni:
        if   t in sig_tickers:    out.add(t)
        elif _nrm(t) in _n2sig:   out.add(_n2sig[_nrm(t)])
    return out

sp500_m  = _match_uni(sp500_tickers)
sp1500_m = _match_uni(sp1500_tickers)
ru3k_m   = ru3k_tickers & sig_tickers   # RU3K already in BESTTICKER format

sp1500_need = sorted((sp500_m | sp1500_m) - fetched_tickers)
ru3k_need   = sorted((ru3k_m - sp1500_m)  - fetched_tickers)

print(f'Universe matches  : SP500={len(sp500_m)}, SP1500={len(sp1500_m)}, RU3K={len(ru3k_m)}')
print(f'Need to fetch     : SP1500={len(sp1500_need):,}, RU3K-only={len(ru3k_need):,}')

# ── Single-ticker helper: tries Yahoo-format variants ─────────────────────────
def _fetch_single(t):
    for v in dict.fromkeys([t, t.replace('.', '-'), t.replace('-', '.')]):
        try:
            h = yf.Ticker(v).history(start=PRICE_START, end=PRICE_END, auto_adjust=True)
            if not h.empty:
                d = h[['Close']].reset_index()
                d.columns = ['date', 'adj_close']
                d['ticker']    = t
                d['date']      = pd.to_datetime(d['date'].values.astype('datetime64[D]'))
                d['adj_close'] = d['adj_close'].astype('float32')
                return d
        except Exception:
            pass
    return None

# ── Streaming Parquet writer helpers ──────────────────────────────────────────
_COLS        = ['date', 'ticker', 'adj_close']   # canonical column order
PRICES_TMP   = DATA_DIR / 'prices_tmp.parquet'
price_writer = None
n_written    = 0
failed_ind   = []

def _write(df):
    global price_writer, n_written
    tbl = pa.Table.from_pandas(df[_COLS], preserve_index=False)  # enforce column order
    if price_writer is None:
        price_writer = pq.ParquetWriter(PRICES_TMP, tbl.schema, compression='snappy')
    price_writer.write_table(tbl)
    n_written += len(df)

def _batch_fetch(batch):
    """Batch yf.download; returns (DataFrame | None, set-of-tickers-got)."""
    try:
        raw = yf.download(batch, start=PRICE_START, end=PRICE_END,
                          auto_adjust=True, progress=False, threads=True)
        if raw.empty:
            return None, set()
        close = (raw['Close'] if isinstance(raw.columns, pd.MultiIndex)
                 else raw[['Close']].rename(columns={'Close': batch[0]}))
        cl = close.stack(future_stack=True).reset_index()
        cl.columns = ['date', 'ticker', 'adj_close']
        cl = cl.dropna(subset=['adj_close'])
        if cl.empty:
            return None, set()
        cl['adj_close'] = cl['adj_close'].astype('float32')
        cl['date']      = pd.to_datetime(cl['date'].values.astype('datetime64[D]'))
        return cl, set(cl['ticker'].unique())
    except Exception:
        return None, set()

# ── Phase 1: SP500 + SP1500  (batch=20, individual retry for misses) ──────────
BATCH_HI = 20
for i in tqdm(range(0, len(sp1500_need), BATCH_HI), desc='SP500/SP1500 prices'):
    batch    = sp1500_need[i : i+BATCH_HI]
    cl, got  = _batch_fetch(batch)
    if cl is not None:
        _write(cl)
    # Individually retry any SP1500 ticker absent from batch result
    for t in batch:
        if t not in got:
            df = _fetch_single(t)
            if df is not None:
                _write(df)
            else:
                failed_ind.append(t)
            time.sleep(0.2)
    time.sleep(0.5)

print(f'SP1500 phase done. Rows written: {n_written:,}.  Failed retries: {len(failed_ind):,}')

# ── Phase 2: RU3K-only  (batch=50, no individual retry — accept some misses) ──
BATCH_LO = 50
for i in tqdm(range(0, len(ru3k_need), BATCH_LO), desc='RU3K prices'):
    batch   = ru3k_need[i : i+BATCH_LO]
    cl, _   = _batch_fetch(batch)
    if cl is not None:
        _write(cl)
    time.sleep(0.5)

print(f'RU3K phase done.  Total rows written: {n_written:,}')
if price_writer:
    price_writer.close()

# ── Merge incremental data into prices.parquet ────────────────────────────────
import shutil
if PRICES_TMP.exists():
    if fetched_tickers:
        old    = pd.read_parquet(PRICES_PARQUET)
        new    = pd.read_parquet(PRICES_TMP)
        merged = pd.concat([old, new], ignore_index=True).drop_duplicates(['date', 'ticker'])
        merged.to_parquet(PRICES_PARQUET, index=False)
        PRICES_TMP.unlink()
        print(f'Merged with existing data.')
    else:
        shutil.move(str(PRICES_TMP), str(PRICES_PARQUET))
elif not fetched_tickers:
    print('WARNING: No prices fetched and no existing file — prices.parquet was not created.')

# ── Coverage report ────────────────────────────────────────────────────────────
if PRICES_PARQUET.exists():
    fin = set(pd.read_parquet(PRICES_PARQUET, columns=['ticker'])['ticker'].unique())
    print(f'\n=== PRICE COVERAGE REPORT ===')
    print(f'Total tickers in prices.parquet : {len(fin):,}')
    for name, uni in [('SP500 ', sp500_m), ('SP1500', sp1500_m), ('RU3K  ', ru3k_m)]:
        n, d = len(uni & fin), len(uni)
        print(f'  {name}: {n:>4}/{d:<4}  ({n/max(1,d):.0%})')

Existing prices.parquet: 3,109 tickers already fetched
Universe matches  : SP500=497, SP1500=1465, RU3K=7877
Need to fetch     : SP1500=0, RU3K-only=4,771


SP500/SP1500 prices: 0it [00:00, ?it/s]

SP1500 phase done. Rows written: 0.  Failed retries: 0


RU3K prices:   0%|          | 0/96 [00:00<?, ?it/s]

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 0190A"}}}


...

['ZRAN']: YFPricesMissingError('possibly delisted; no price data found  (1d 2009-12-01 -> 2026-05-01)')


['ZMTP', 'ZYXI', 'ZOM', 'ZUO', 'ZOLL', 'ZNGA', 'ZZ', 'ZPTA', 'ZYNE', 'ZYME', 'ZOLT', 'ZSPC', 'ZPIN', 'ZUMZ', 'ZMH', 'ZS', 'ZTO', 'ZY', 'ZU']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


RU3K phase done.  Total rows written: 779,119


Merged with existing data.



=== PRICE COVERAGE REPORT ===
Total tickers in prices.parquet : 3,429
  SP500 :  497/497   (100%)
  SP1500: 1465/1465  (100%)
  RU3K  : 3426/7877  (43%)


## 8. Load Signal Data (Total Slice) + Entry Date Logic

**Look-ahead rule (documented):**
- `MOSTIMPORTANTDATEUTC` hour **≥ 16 UTC** → After-market-close (AMC) → entry = **next** NYSE trading day close  
- `MOSTIMPORTANTDATEUTC` hour **< 16 UTC** → Before-market-open or intraday → entry = **same** NYSE trading day close  

This is conservative: any call during market hours (13–16 UTC = 8–11 AM ET) is treated as tradeable same-day.

In [9]:
# Stream through Parquet, keep only Total rows (memory-efficient)
print('Loading Total-slice rows from signals.parquet...')

pf     = pq.ParquetFile(SIGNALS_PARQUET)
chunks = []

for batch in pf.iter_batches(batch_size=50_000):
    df_batch = batch.to_pandas()
    df_filt  = df_batch[df_batch['SignalType'] == 'Total']
    if len(df_filt) > 0:
        chunks.append(df_filt)

df_total = pd.concat(chunks, ignore_index=True)
del chunks
print(f'Total-slice rows: {len(df_total):,}  |  columns: {df_total.shape[1]}')

Loading Total-slice rows from signals.parquet...


Total-slice rows: 376,790  |  columns: 447


In [10]:
# Parse timestamps and classify AMC vs BMO
# format='mixed' handles the mix of "YYYY-MM-DD HH:MM:SS" and date-only "YYYY-MM-DD" rows
df_total['dt_utc']        = pd.to_datetime(df_total['MOSTIMPORTANTDATEUTC'], utc=True, format='mixed')
df_total['call_date']     = df_total['dt_utc'].dt.normalize().dt.tz_localize(None)
df_total['call_hour_utc'] = df_total['dt_utc'].dt.hour
df_total['is_amc']        = df_total['call_hour_utc'] >= 16

# §3.7 audit: inspect INGESTDATEUTC lag
# This dataset was built by batch-ingesting historical transcripts: mean lag ≈ 4.5 years.
# INGESTDATEUTC does NOT represent live data-availability for historical calls.
# Design choice (documented per §3.7): use MOSTIMPORTANTDATEUTC-based entry only.
df_total['dt_ingest']   = pd.to_datetime(df_total['INGESTDATEUTC'], utc=True, format='mixed', errors='coerce')
df_total['ingest_date'] = df_total['dt_ingest'].dt.normalize().dt.tz_localize(None)

print('Call hour (UTC) distribution:')
print(df_total['call_hour_utc'].value_counts().sort_index())
print(f'\nAMC (≥16 UTC, next-day entry): {df_total["is_amc"].sum():,}  ({df_total["is_amc"].mean():.1%})')
print(f'BMO (<16 UTC, same-day entry): {(~df_total["is_amc"]).sum():,}  ({(~df_total["is_amc"]).mean():.1%})')
print(f'\nNote: date-only rows (hour=0) are treated as BMO — same-day entry.')

ingest_lag = (df_total['ingest_date'] - df_total['call_date']).dt.days
print(f'\n§3.7 — INGESTDATEUTC lag vs call_date (days):')
print(ingest_lag.describe().to_string())
print(f'Conclusion: mean lag {ingest_lag.mean():.0f}d confirms batch historical backfill.')
print(f'Entry date uses MOSTIMPORTANTDATEUTC only (INGESTDATEUTC not a valid real-time constraint).')

df_total.drop(columns=['dt_ingest', 'ingest_date'], inplace=True)

Call hour (UTC) distribution:
call_hour_utc
0     10700
1      3978
2      1627
3      1136
4      1829
5      3080
6      7623
7     13447
8     18126
9     13757
10     9901
11     7971
12    41922
13    47058
14    44619
15    40253
16    17135
17     6941
18     4791
19     2360
20    22314
21    37210
22    15700
23     3312
Name: count, dtype: int64

AMC (≥16 UTC, next-day entry): 109,763  (29.1%)
BMO (<16 UTC, same-day entry): 267,027  (70.9%)

Note: date-only rows (hour=0) are treated as BMO — same-day entry.

§3.7 — INGESTDATEUTC lag vs call_date (days):
count   376790.0000
mean      1657.7820
std       1545.2920
min         -7.0000
25%        103.0000
50%       1259.0000
75%       2914.0000
max       4931.0000
Conclusion: mean lag 1658d confirms batch historical backfill.
Entry date uses MOSTIMPORTANTDATEUTC only (INGESTDATEUTC not a valid real-time constraint).


In [11]:
# Vectorized entry-date assignment using NYSE calendar
call_dates_arr = df_total['call_date'].values.astype('datetime64[D]')
is_amc         = df_total['is_amc'].values

# searchsorted 'left' : same trading day if it exists, else next
# searchsorted 'right': always next trading day (for AMC)
idx_left  = np.searchsorted(trading_days_arr, call_dates_arr, side='left')
idx_right = np.searchsorted(trading_days_arr, call_dates_arr, side='right')

idx_bmo   = np.clip(idx_left,  0, len(trading_days_arr) - 1)
idx_amc   = np.clip(idx_right, 0, len(trading_days_arr) - 1)

entry_idx = np.where(is_amc, idx_amc, idx_bmo)
df_total['entry_date'] = pd.DatetimeIndex(trading_days_arr[entry_idx].astype('datetime64[ns]'))

# ── LOOK-AHEAD ASSERTIONS ──────────────────────────────────────────────────────
amc_mask = df_total['is_amc']
assert (df_total.loc[amc_mask,  'entry_date'] >  df_total.loc[amc_mask,  'call_date']).all(), \
    'FAIL: AMC events must enter after call date'
assert (df_total.loc[~amc_mask, 'entry_date'] >= df_total.loc[~amc_mask, 'call_date']).all(), \
    'FAIL: BMO events must enter on or after call date'

print('Look-ahead assertions passed.')
print(df_total[['DocDate','call_hour_utc','is_amc','entry_date']].head(8).to_string())

df_total.drop(columns=['dt_utc', 'call_date', 'call_hour_utc', 'is_amc'], inplace=True)

Look-ahead assertions passed.
      DocDate  call_hour_utc  is_amc entry_date
0  2010-01-04             22    True 2010-01-05
1  2010-01-05             15   False 2010-01-05
2  2010-01-05             21    True 2010-01-06
3  2010-01-05             22    True 2010-01-06
4  2010-01-05             22    True 2010-01-06
5  2010-01-06             15   False 2010-01-06
6  2010-01-06             14   False 2010-01-06
7  2010-01-06             14   False 2010-01-06


## 9. Forward Return Computation

Returns are computed **after** entry_date is determined — they are targets, never features.

In [12]:
# Load prices
prices = pd.read_parquet(PRICES_PARQUET)
prices['date'] = pd.to_datetime(prices['date']).dt.normalize()
print(f'Prices shape: {prices.shape}  |  unique tickers: {prices["ticker"].nunique():,}')
print(f'Price date range: {prices["date"].min().date()} → {prices["date"].max().date()}')

Prices shape: (10071093, 3)  |  unique tickers: 3,429
Price date range: 2009-12-01 → 2026-04-30


In [13]:
HORIZONS = [1, 3, 5, 10, 20]

# Map entry_date to its index in trading_days_arr
entry_dates_d64   = df_total['entry_date'].values.astype('datetime64[D]')
entry_td_idx      = np.searchsorted(trading_days_arr, entry_dates_d64, side='left')

# Compute exit dates for each horizon
for h in HORIZONS:
    exit_idx = np.clip(entry_td_idx + h, 0, len(trading_days_arr) - 1)
    df_total[f'exit_date_{h}d'] = pd.DatetimeIndex(
        trading_days_arr[exit_idx].astype('datetime64[ns]')
    )

# ── LOOK-AHEAD ASSERTION: exit dates must be strictly after entry ──────────────
for h in HORIZONS:
    assert (df_total[f'exit_date_{h}d'] > df_total['entry_date']).all(), \
        f'FAIL: exit_date_{h}d not strictly after entry_date'
print('Exit-date assertions passed.')

Exit-date assertions passed.


In [14]:
# Join entry price
df_total = df_total.merge(
    prices.rename(columns={'date': 'entry_date', 'ticker': 'BESTTICKER', 'adj_close': 'price_entry'}),
    on=['BESTTICKER', 'entry_date'], how='left'
)

# Join exit prices and compute returns
for h in HORIZONS:
    df_total = df_total.merge(
        prices.rename(columns={
            'date': f'exit_date_{h}d', 'ticker': 'BESTTICKER', 'adj_close': f'px_exit_{h}d'
        }),
        on=['BESTTICKER', f'exit_date_{h}d'], how='left'
    )
    df_total[f'return_{h}d'] = (df_total[f'px_exit_{h}d'] / df_total['price_entry'] - 1).astype('float32')
    df_total.drop(columns=[f'px_exit_{h}d', f'exit_date_{h}d'], inplace=True)

df_total.drop(columns=['price_entry'], inplace=True)

# Winsorize at 0.1% / 99.9% — expanding-window roll-forward, look-ahead-free (§3.9)
# For each quarter Q, bounds are computed from all events in prior quarters only.
# The first quarter with fewer than 50 valid returns uses its own distribution (cold-start).
# Quarterly steps align with the walk-forward framework and are finer than annual steps.
print('Winsorizing returns (quarterly expanding-window, look-ahead-free):')
df_total['_qtr'] = pd.to_datetime(df_total['entry_date']).dt.to_period('Q')
qtrs = sorted(df_total['_qtr'].unique())
print(f'  Quarters: {qtrs[0]} → {qtrs[-1]}  ({len(qtrs)} steps)')

for h in HORIZONS:
    col = f'return_{h}d'
    clipped = df_total[col].copy()
    n_clipped_total = 0

    for qtr in qtrs:
        past = df_total[df_total['_qtr'] < qtr][col].dropna()
        if len(past) < 50:          # cold-start: too little history, use own quarter
            past = df_total[df_total['_qtr'] == qtr][col].dropna()
        lo, hi = past.quantile(0.001), past.quantile(0.999)
        mask = df_total['_qtr'] == qtr
        n_clip = ((clipped[mask] < lo) | (clipped[mask] > hi)).sum()
        clipped.loc[mask] = clipped.loc[mask].clip(lo, hi)
        n_clipped_total += n_clip

    df_total[col] = clipped.astype('float32')
    valid = df_total[col].dropna()
    print(f'  return_{h:>2}d : clipped {n_clipped_total:>4} rows  [{valid.min():.4f}, {valid.max():.4f}]')

df_total.drop(columns=['_qtr'], inplace=True)

# Coverage report
print('\nForward return coverage (fraction of events with valid prices):')
for h in HORIZONS:
    n = df_total[f'return_{h}d'].notna().sum()
    print(f'  return_{h:>2}d : {n:>7,}  ({n/len(df_total):.1%})')

Winsorizing returns (quarterly expanding-window, look-ahead-free):
  Quarters: 2010Q1 → 2026Q2  (66 steps)


  return_ 1d : clipped  436 rows  [-0.1987, 0.2567]


  return_ 3d : clipped  453 rows  [-0.3053, 0.4089]


  return_ 5d : clipped  433 rows  [-0.3552, 0.5199]


  return_10d : clipped  495 rows  [-0.4913, 0.9228]


  return_20d : clipped  645 rows  [-0.6530, 1.0525]

Forward return coverage (fraction of events with valid prices):
  return_ 1d : 129,320  (34.3%)
  return_ 3d : 129,316  (34.3%)
  return_ 5d : 129,315  (34.3%)
  return_10d : 129,312  (34.3%)
  return_20d : 129,309  (34.3%)


## 9b. Corporate-Action / No-Price Rule  *(audit checklist §3.9)*

**NaN-price exclusion:** Events where `price_entry` is NaN (ticker delisted, suspended, or not in the price cache) produce NaN forward returns and are excluded from all downstream IC and portfolio calculations. No roll-forward to the next valid trading day is applied.

**Coverage note:** ~68% of Total-slice rows are non-US or non-universe tickers for which no price was fetched. Within each target universe (SP500 / SP1500 / RU3K), coverage is >97%.

**Winsorization (look-ahead-free, quarterly roll-forward):** Returns are clipped at the 0.1th / 99.9th percentile per horizon using a quarterly expanding-window design. For each quarter Q, clip bounds are computed from all events in prior quarters only — no future return data ever informs the clipping of a past event. The first quarter with fewer than 50 valid returns (cold-start) uses its own distribution. Quarterly granularity aligns with the walk-forward framework. NaN rows pass through `.clip()` unchanged.

In [15]:
# ── LOOK-AHEAD ASSERTION: forward return columns must NOT appear in at_cols_keep ─
return_cols = {f'return_{h}d' for h in HORIZONS}
feature_cols_check = set(at_cols_keep) | {'ATCClassifierScore',
    'EventsScore_1_1_1', 'EventsScore_4_2_1', 'EventsScore_3_1_0', 'EventsScore_1_1_0'}
assert return_cols.isdisjoint(feature_cols_check), 'FAIL: return columns leaked into feature set'
print('Return-columns-not-in-features assertion passed.')

Return-columns-not-in-features assertion passed.


## 10. Universe Membership Flags

In [16]:
df_total['in_SP500']  = df_total['BESTTICKER'].isin(sp500_tickers)
df_total['in_SP1500'] = df_total['BESTTICKER'].isin(sp1500_tickers)
df_total['in_RU3K']   = df_total['BESTTICKER'].isin(ru3k_tickers)

has_return = df_total['return_5d'].notna()
print('Events with valid 5d return by universe:')
for u in ['SP500', 'SP1500', 'RU3K']:
    n = (df_total[f'in_{u}'] & has_return).sum()
    print(f'  {u:6s}: {n:>7,}')

Events with valid 5d return by universe:
  SP500 :  29,946
  SP1500:  78,652
  RU3K  : 129,289


## 11. Feature Engineering

**Best practice — Aspect × Theme cross-product features (prefix: `at_`):**
For each of the 5×9 = 45 (Aspect, Theme) pairs, sum sentence counts over all
magnitudes and compute Positive, Negative, total, and net sentiment.
This preserves the cross-product structure so the model can distinguish
`Forecast×FinancialPerformance` (forward-looking) from
`CurrentState×FinancialPerformance` (backward-looking).

Plus raw EventScores and ATCClassifierScore (13 raw-score columns).

QoQ / 2Q / YoY deltas computed via `shift(1/2/4)` within each ticker sorted by `entry_date` — uses only past data.


In [17]:
# Derive dimension values from actual metadata (robust, no hardcoding)
VALID_ASPECTS = [a for a in sorted(at_meta['aspect'].unique()) if a not in ('Fluff', 'Filler')]
THEMES        = sorted(at_meta['theme'].unique())
SENTIMENTS    = sorted(at_meta['sentiment'].unique())   # Negative, Neutral, Positive
MAGNITUDES    = sorted(at_meta['magnitude'].unique())   # High, Low, Medium

print(f'Aspects   : {VALID_ASPECTS}')
print(f'Themes    : {THEMES}')
print(f'Sentiments: {SENTIMENTS}')
print(f'Magnitudes: {MAGNITUDES}')

feat = df_total[[
    'DocDate', 'BESTTICKER', 'SECTOR', 'QTR_YEAR', 'entry_date',
    'in_SP500', 'in_SP1500', 'in_RU3K',
    'ATCClassifierScore',
    'EventsScore_1_1_1', 'EventPos_1_1_1', 'EventNeg_1_1_1',
    'EventsScore_4_2_1', 'EventPos_4_2_1', 'EventNeg_4_2_1',
    'EventsScore_3_1_0', 'EventPos_3_1_0', 'EventNeg_3_1_0',
    'EventsScore_1_1_0', 'EventPos_1_1_0', 'EventNeg_1_1_0',
] + [f'return_{h}d' for h in HORIZONS]].copy()

# ── Aspect × Theme cross-product features  (prefix: at_)  ────────────────────
# Magnitude is collapsed (summed) — it encodes degree, not direction.
# Preserving the Aspect×Theme cross-product lets models distinguish, e.g.,
#   Forecast×FinancialPerformance  vs  CurrentState×FinancialPerformance.
for asp in VALID_ASPECTS:
    for theme in THEMES:
        pos_cols = at_meta.loc[
            (at_meta['aspect'] == asp) & (at_meta['theme'] == theme) &
            (at_meta['sentiment'] == 'Positive'), 'col'].tolist()
        neg_cols = at_meta.loc[
            (at_meta['aspect'] == asp) & (at_meta['theme'] == theme) &
            (at_meta['sentiment'] == 'Negative'), 'col'].tolist()
        neu_cols = at_meta.loc[
            (at_meta['aspect'] == asp) & (at_meta['theme'] == theme) &
            (at_meta['sentiment'] == 'Neutral'), 'col'].tolist()

        pos = (df_total[pos_cols].sum(axis=1).astype('float32')
               if pos_cols else pd.Series(0.0, index=feat.index, dtype='float32'))
        neg = (df_total[neg_cols].sum(axis=1).astype('float32')
               if neg_cols else pd.Series(0.0, index=feat.index, dtype='float32'))
        neu = (df_total[neu_cols].sum(axis=1).astype('float32')
               if neu_cols else pd.Series(0.0, index=feat.index, dtype='float32'))
        tot = pos + neg + neu

        feat[f'at_{asp}_{theme}_Positive']      = pos
        feat[f'at_{asp}_{theme}_Negative']      = neg
        feat[f'at_{asp}_{theme}_total']         = tot.astype('float32')
        feat[f'at_{asp}_{theme}_net_sentiment'] = ((pos - neg) / (tot + 1)).astype('float32')

# ── Save raw AspectTheme sparse matrix for stretch model  ─────────────────────
SPARSE_PARQUET = DATA_DIR / 'sparse_features.parquet'
if not SPARSE_PARQUET.exists():
    df_total[['BESTTICKER', 'entry_date'] + at_cols_keep].to_parquet(SPARSE_PARQUET, index=False)
    print(f'Saved sparse_features.parquet ({SPARSE_PARQUET.stat().st_size / 1e6:.0f} MB)')
else:
    print(f'sparse_features.parquet exists ({SPARSE_PARQUET.stat().st_size / 1e6:.0f} MB) — skipping.')

del df_total

cross_pairs = len(VALID_ASPECTS) * len(THEMES)          # 5 × 9 = 45
cross_feats = cross_pairs * 4                            # Positive, Negative, total, net
raw_scores  = 13                                         # ATCClassifierScore + 4 variants × 3
eng_feat_cols = [c for c in feat.columns if c not in [
    'DocDate','BESTTICKER','SECTOR','QTR_YEAR','entry_date',
    'in_SP500','in_SP1500','in_RU3K'
] and not c.startswith('return_')]
print(f'\nAspect × Theme pairs   : {cross_pairs}  ({len(VALID_ASPECTS)} aspects × {len(THEMES)} themes)')
print(f'Cross-product features : {cross_feats}  (pairs × 4: Pos/Neg/total/net)')
print(f'Raw score features     : {raw_scores}')
print(f'Total base features    : {len(eng_feat_cols)}')
print('Sample names:', [c for c in eng_feat_cols if 'at_' in c][:4], '...')


Aspects   : ['CurrentState', 'Forecast', 'Other', 'StrategicPosition', 'Surprise']
Themes    : ['CapitalAllocation', 'ESG', 'FinancialPerformance', 'MacroeconomicFactors', 'MarketAndCompetitivePosition', 'OperationalPerformance', 'Other', 'RegulatoryAndLegalIssues', 'StrategicInitiatives']
Sentiments: ['Negative', 'Neutral', 'Positive']
Magnitudes: ['High', 'Low', 'Medium']


sparse_features.parquet exists (44 MB) — skipping.

Aspect × Theme pairs   : 45  (5 aspects × 9 themes)
Cross-product features : 180  (pairs × 4: Pos/Neg/total/net)
Raw score features     : 13
Total base features    : 193
Sample names: ['at_CurrentState_CapitalAllocation_Positive', 'at_CurrentState_CapitalAllocation_Negative', 'at_CurrentState_CapitalAllocation_total', 'at_CurrentState_CapitalAllocation_net_sentiment'] ...


In [18]:
# ── Lagged trend features: QoQ (1Q), 2-quarter, and YoY (4Q) ─────────────────
# All shifts look backward in time (past data only).
# feat is sorted by (BESTTICKER, entry_date) before shifting.
feat.sort_values(['BESTTICKER', 'entry_date'], inplace=True)
feat.reset_index(drop=True, inplace=True)

delta_source_cols = [c for c in eng_feat_cols if not c.startswith('return_')]

for lag, suffix in [(1, '_qoq'), (2, '_2q'), (4, '_yoy')]:
    prev     = feat.groupby('BESTTICKER')[delta_source_cols].shift(lag)
    delta_df = (feat[delta_source_cols] - prev).astype('float32')
    delta_df.columns = [f'{c}{suffix}' for c in delta_source_cols]
    feat = pd.concat([feat, delta_df], axis=1)
    del delta_df, prev

all_feat_cols = [c for c in feat.columns if c not in [
    'DocDate','BESTTICKER','SECTOR','QTR_YEAR','entry_date',
    'in_SP500','in_SP1500','in_RU3K'
] and not c.startswith('return_')]
print(f'Total features (base + QoQ + 2Q + YoY): {len(all_feat_cols)}')
print(f'Feature DataFrame shape: {feat.shape}')
print(f'Memory usage: {feat.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Total features (base + QoQ + 2Q + YoY): 772
Feature DataFrame shape: (376790, 785)
Memory usage: 1.27 GB


## 12. Save Master Parquet

In [19]:
EVENTS_PARQUET = DATA_DIR / 'events_with_returns.parquet'
feat.to_parquet(EVENTS_PARQUET, index=False)
print(f'Saved events_with_returns.parquet')
print(f'  Shape  : {feat.shape}')
print(f'  Size   : {EVENTS_PARQUET.stat().st_size / 1e6:.0f} MB')
print()
print('Columns summary:')
print(f'  Metadata : DocDate, BESTTICKER, SECTOR, QTR_YEAR, entry_date, in_SP500/1500/RU3K')
print(f'  Returns  : return_1d/3d/5d/10d/20d  (targets — not features)')
print(f'  Features : {len(all_feat_cols)} columns  (at_*cross-product, EventScores, ATCClassifierScore + QoQ/2Q/YoY trends)')
print()
print('Sample rows:')
display(feat[['DocDate','BESTTICKER','SECTOR','entry_date',
              'ATCClassifierScore','return_1d','return_5d',
              'at_Forecast_FinancialPerformance_net_sentiment',
              'at_CurrentState_FinancialPerformance_net_sentiment']].head())


Saved events_with_returns.parquet
  Shape  : (376790, 785)
  Size   : 261 MB

Columns summary:
  Metadata : DocDate, BESTTICKER, SECTOR, QTR_YEAR, entry_date, in_SP500/1500/RU3K
  Returns  : return_1d/3d/5d/10d/20d  (targets — not features)
  Features : 772 columns  (at_*cross-product, EventScores, ATCClassifierScore + QoQ/2Q/YoY trends)

Sample rows:


,DocDate,BESTTICKER,SECTOR,entry_date,ATCClassifierScore,return_1d,return_5d,at_Forecast_FinancialPerformance_net_sentiment,at_CurrentState_FinancialPerformance_net_sentiment
0,2021-08-20,000001,Financials,2021-08-20,0.0347,NaN,NaN,0.5000,0.6741
1,2022-03-10,000001,Financials,2022-03-10,0.0273,NaN,NaN,0.4815,0.6753
2,2022-08-18,000001,Financials,2022-08-18,-0.0056,NaN,NaN,0.5000,0.6127
3,2024-08-16,000001,Financials,2024-08-16,-0.0341,NaN,NaN,0.5714,0.3261
4,2025-03-17,000001,Financials,2025-03-17,0.0275,NaN,NaN,0.3906,0.4035


In [20]:
# ── 13. Signal-Slice Score Cache ──────────────────────────────────────────────
# Saves ATCClassifierScore + EventScores for Total / CEO / CFO / Analysts slices.
# Required by §2.2: IC and quintile comparison across SignalType speaker cuts.
# Returns are NOT stored here — join via (BESTTICKER, entry_date) against
# events_with_returns.parquet in notebook 01.

SLICES_PARQUET  = DATA_DIR / 'signal_slices.parquet'
COMPARE_SLICES  = ['Total', 'CEO', 'CFO', 'Analysts']
SCORE_COLS_KEEP = [
    'ATCClassifierScore',
    'EventPos_1_1_1', 'EventNeg_1_1_1', 'EventsScore_1_1_1',
    'EventPos_4_2_1', 'EventNeg_4_2_1', 'EventsScore_4_2_1',
    'EventPos_3_1_0', 'EventNeg_3_1_0', 'EventsScore_3_1_0',
    'EventPos_1_1_0', 'EventNeg_1_1_0', 'EventsScore_1_1_0',
]
SLICE_META = ['SignalType', 'DocDate', 'BESTTICKER', 'SECTOR', 'QTR_YEAR',
              'MOSTIMPORTANTDATEUTC']

if SLICES_PARQUET.exists():
    print(f'signal_slices.parquet exists — skipping.')
else:
    pf2     = pq.ParquetFile(SIGNALS_PARQUET)
    chunks2 = []
    for batch in pf2.iter_batches(batch_size=50_000):
        df_b = batch.to_pandas()
        df_b = df_b[df_b['SignalType'].isin(COMPARE_SLICES)]
        if len(df_b):
            chunks2.append(df_b[SLICE_META + SCORE_COLS_KEEP])
    df_sl = pd.concat(chunks2, ignore_index=True)
    del chunks2
    print(f'Loaded {len(df_sl):,} rows  ({COMPARE_SLICES})')

    # ── Entry date (same AMC/BMO rule as Total; no INGESTDATEUTC adjustment) ──
    df_sl['dt_utc']    = pd.to_datetime(df_sl['MOSTIMPORTANTDATEUTC'], utc=True, format='mixed')
    df_sl['call_date'] = df_sl['dt_utc'].dt.normalize().dt.tz_localize(None)
    df_sl['is_amc_sl'] = df_sl['dt_utc'].dt.hour >= 16

    cd   = df_sl['call_date'].values.astype('datetime64[D]')
    amc  = df_sl['is_amc_sl'].values
    il   = np.searchsorted(trading_days_arr, cd, side='left')
    ir   = np.searchsorted(trading_days_arr, cd, side='right')
    e    = np.where(amc, np.clip(ir, 0, len(trading_days_arr)-1),
                         np.clip(il, 0, len(trading_days_arr)-1))
    df_sl['entry_date'] = pd.DatetimeIndex(trading_days_arr[e].astype('datetime64[ns]'))

    # ── Universe membership ───────────────────────────────────────────────────
    df_sl['in_SP500']  = df_sl['BESTTICKER'].isin(sp500_tickers)
    df_sl['in_SP1500'] = df_sl['BESTTICKER'].isin(sp1500_tickers)
    df_sl['in_RU3K']   = df_sl['BESTTICKER'].isin(ru3k_tickers)

    df_sl = df_sl.drop(columns=['dt_utc', 'call_date', 'is_amc_sl'])
    df_sl.to_parquet(SLICES_PARQUET, index=False)
    print(f'Saved signal_slices.parquet  ({SLICES_PARQUET.stat().st_size/1e6:.1f} MB)')
    print(f'  Shape: {df_sl.shape}')
    for st in COMPARE_SLICES:
        n = (df_sl['SignalType'] == st).sum()
        print(f'  {st:10s}: {n:,} rows')

signal_slices.parquet exists — skipping.


## 13. Final Sanity Checks

In [21]:
print('=== DATA PREP SUMMARY ===')
print(f'Events (Total slice): {len(feat):,}')
print(f'Date range : {feat["DocDate"].min()} → {feat["DocDate"].max()}')
print(f'Tickers    : {feat["BESTTICKER"].nunique():,}')
print(f'Sectors    : {feat["SECTOR"].nunique()}')
print()
print('Return stats (5d horizon):')
print(feat['return_5d'].describe())
print()

# Final look-ahead assertions on the saved file
ret_cols = {f'return_{h}d' for h in HORIZONS}
feat_set = set(all_feat_cols)
assert ret_cols.isdisjoint(feat_set), 'FAIL: return cols leaked into features'

# entry_date must be a NYSE trading day
entry_d64 = feat['entry_date'].values.astype('datetime64[D]')
assert np.all(np.isin(entry_d64, trading_days_arr)), 'FAIL: some entry dates are not NYSE trading days'

print('All look-ahead assertions passed. Notebook 00 complete.')
print()
print('Files written to data/:')
for p in sorted(DATA_DIR.iterdir()):
    print(f'  {p.name:40s} {p.stat().st_size / 1e6:>8.1f} MB')

=== DATA PREP SUMMARY ===
Events (Total slice): 376,790
Date range : 2010-01-04 → 2026-04-21
Tickers    : 17,636
Sectors    : 11

Return stats (5d horizon):
count   129315.0000
mean         0.0018
std          0.0695
min         -0.3552
25%         -0.0305
50%          0.0008
75%          0.0321
max          0.5199
Name: return_5d, dtype: float64

All look-ahead assertions passed. Notebook 00 complete.

Files written to data/:
  events_with_returns.parquet                 260.7 MB
  prices.parquet                               45.4 MB
  signal_slices.parquet                        37.1 MB
  signals.parquet                             317.8 MB
  sparse_features.parquet                      44.0 MB
  universes.json                                0.1 MB
